In [1]:

from pathlib import Path
import time

import numpy as np
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Binarizer, OneHotEncoder, PCA, SQLTransformer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# keep the pandas tables wide enough to read in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# store the project paths, the watched stream folder, the stream checkpoint, and the shared seed in one place.
PROJECT_DIR = Path('/Users/alexdevoid/Documents/Stats/ST554-HW/FinalProject')

# start Spark session
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('final_project_power_stream')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
spark.conf.set('spark.sql.shuffle.partitions', '8')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/24 23:11:50 WARN Utils: Your hostname, Alexs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.101 instead (on interface en0)
26/04/24 23:11:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 23:11:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:

# read data from URL into pandas
power_pdf = pd.read_csv('https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv')

# convert into a cached Spark SQL DataFrame.
power_sdf = spark.createDataFrame(power_pdf).cache()
_ = power_sdf.count()

# summarize the row count, column count, and response column
data_summary = pd.DataFrame(
    {
        'rows': [power_pdf.shape[0]],
        'columns': [power_pdf.shape[1]],
        'response': ['Power_Zone_3'],
    }
)


# show summary
display(data_summary)
# how first rows of the Spark SQL DataFrame
display(power_pdf.head())

,rows,columns,response
0,47174,10,Power_Zone_3


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0
